<a href="https://colab.research.google.com/github/mabravoreyes/analysis/blob/main/mongolia_traceroute_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mongolia School Traceroute Explorer

This notebook lets you explore HERMES-enriched traceroute data from M-Lab speed tests
originating at GIGA-connected schools in Mongolia.

**Data periods:**
- **October 2025**
- **November 2025**

**No BigQuery access needed** — the parquet files are loaded directly.

---

## 0. Setup

In [ ]:
# If running on Google Colab, uncomment the following lines:
!pip install db-dtypes plotly scikit-learn geopy
from google.colab import files
# Upload the parquet files when prompted:
uploaded = files.upload()

In [ ]:
import db_dtypes
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)

## 1. Load Data

Place the parquet files in the same directory as this notebook (or adjust the paths below).

| File | Period | Rows | Description |
|------|--------|-----:|-------------|
| `hermes_broad_MN_2025-10-01_to_2025-10-31.parquet` | Nov 2025| 37,743 | HERMES giga-meter measurements |
| `giga_2025-10-15_to_2025-11-15.parquet` | Oct 2025 | 57,649 | Broad HERMES (IP-matched) |

In [ ]:
# Adjust these paths for your environment
PATH_OCT = "hermes_broad_MN_2025-10-01_to_2025-10-31.parquet"

df_oct = pd.read_parquet(PATH_OCT)

print(f"October dataset:   {len(df_oct):,} rows")

October dataset:   57,649 rows


## 2. Data Dictionary

### Measurement-level columns

Each row is one NDT speed test from a school, enriched by
traceroute data.

| Column | Type | Description |
|--------|------|-------------|
| `id` | string | Unique measurement ID |
| `partition_date` | date | Date of measurement |
| `client_ip` | string | Source (school) IP address |
| `src_country` | string | Source country code (always "MN") |
| `src_state` | string | Source province/state |
| `src_lat`, `src_lon` | float | Source geolocation (school) |
| `src_city` | string | Source city |
| `src_asn` | int | Source autonomous system number |
| `src_asn_name` | string | Source ASN organization name |
| `dst_country` | string | Destination country code |
| `dst_lat`, `dst_lon` | float | Destination server geolocation |
| `dst_city` | string | Destination city |
| `dst_site` | string | M-Lab server site name (e.g. `uln10076`, `icn02`) |
| `dst_asn` | int | Destination ASN |
| `ndt_rtt` | float | NDT minimum RTT (ms) |
| `ndt_throughput` | float | NDT throughput (Mbps) |
| `ndt_loss_rate` | float | NDT packet loss rate (0–1) |
| `forward_updated_node_details` | list\<struct\> | **Forward traceroute hops** (server → client) |
| `reverse_updated_node_details` | list\<struct\> | Reverse traceroute hops (mostly null) |
| `forward_distance` | float | Total geographic distance of forward path (km) |
| `reverse_distance` | float | Total geographic distance of reverse path (km) |
| `is_reaching_dst_asn` | bool | Did the traceroute reach the destination ASN? |

### Hop-level fields (inside `forward_updated_node_details`)

Each hop is a dict with these keys:

| Key | Type | Description |
|-----|------|-------------|
| `ttl` | int | Time-to-live (hop number from server) |
| `addr` | string | IP address of the hop (or `*` for timeout) |
| `associated_asn` | int | ASN this hop belongs to |
| `associated_org` | string | Organization name |
| `associated_peeringdb_name` | string | PeeringDB name (often cleaner) |
| `associated_ixp` | string | IXP name if this hop is at an IXP |
| `cc` | string | Country code (IP geolocation) |
| `place` | string | City/location name |
| `metro` | string | Metro area |
| `latitude`, `longitude` | float | Hop geolocation |
| `rtts` | float | Round-trip time to this hop (ms) |
| `rdns_name` | string | Reverse DNS name |
| `cumulative_distance_km` | float | Cumulative geographic distance from server |
| `distance_to_destination_km` | float | Remaining distance to client |
| `speed_of_internet_fiber` | float | Theoretical fiber speed RTT (ms) |
| `above_baseline_flag` | string | Whether RTT is above expected baseline |
| `increasing_latency_flag` | string | Whether latency is increasing unexpectedly |
| `distance_rtt_check` | string | Consistency check between distance and RTT |
| `facilities_info` | list | Colocation facility details |

### Key destination servers

| Site | Location | ASN | Role |
|------|----------|-----|------|
| `uln10076` | Ulaanbaatar, Mongolia | AS10076 (ERDEMNET) | Domestic server |
| `icn02` | Seoul, South Korea | AS396982 (Google) | International server |
| `lhe152605` | Lahore, Pakistan | AS152605 | International server (Oct only) |

## 3. Quick Exploration

In [ ]:
df = df_oct.copy()

print(f"Date range: {df['partition_date'].min()} to {df['partition_date'].max()}")
print(f"Unique source IPs: {df['client_ip'].nunique()}")
print(f"Unique source ASNs: {df['src_asn'].nunique()}")
print()
print("Destination servers:")
print(df['dst_site'].value_counts().to_string())
print()
print("Source ASNs:")
print(df.groupby('src_asn')['src_asn_name'].first().to_string())

Date range: 2025-10-01 to 2025-10-31
Unique source IPs: 1915
Unique source ASNs: 10

Destination servers:
dst_site
icn02        33388
uln10076     23954
lhe152605      307

Source ASNs:
src_asn
9484                                 Mobinet ISP, MobiCom Corporation
9934                                                 Mongolia Telecom
10076      ERDEMNET Mongolian National Research and Education Network
10219                                        SKYMEDIA CORPORATION LLC
14593                      Space Exploration Technologies Corporation
17882     The first E-commerce and TriplePlay Service ISP in Mongo...
24559                                            G-Mobile Corporation
38805     STXCitinet, Leading Internet & VOIP Service Provider, Ul...
55805                                   MobiCom Corporation, Mongolia
212238                                                           None


In [ ]:
# Preview one row (excluding the large hop-details columns)
df.drop(columns=['forward_updated_node_details', 'reverse_updated_node_details']).head(3)

,id,partition_date,client_ip,src_country,src_state,src_lat,src_lon,src_city,src_asn,src_asn_name,dst_country,dst_lat,dst_lon,dst_city,dst_site,dst_asn,ndt_rtt,ndt_throughput,ndt_loss_rate,forward_distance,reverse_distance,is_reaching_dst_asn
0,m-lab_1758808245_0000000000149F7B,2025-10-09,202.5.201.19,MN,1,47.9094,106.8819,Ulan Bator-1-MN,10076,ERDEMNET Mongolian National Research and Education Network,PK,31.5216,74.4036,Lahore,lhe152605,152605,198.323,20.076048,0.333292,8750.456380,3284.793983,True
1,m-lab_1758808245_000000000014F044,2025-10-09,202.5.199.254,MN,1,47.9094,106.8819,Ulan Bator-1-MN,10076,ERDEMNET Mongolian National Research and Education Network,PK,31.5216,74.4036,Lahore,lhe152605,152605,260.233,10.088491,0.497290,15152.064377,3284.793983,True
2,m-lab_1758808245_0000000000153EB2,2025-10-08,202.5.201.37,MN,1,47.9094,106.8819,Ulan Bator-1-MN,10076,ERDEMNET Mongolian National Research and Education Network,PK,31.5216,74.4036,Lahore,lhe152605,152605,202.594,77.270380,0.000000,8750.456380,3284.793983,True


## 4. Helper Functions

These extract useful information from the raw hop-details arrays.

In [ ]:
def iter_nodes(node_details):
    """Safely iterate over hop details (handles None/NaN/ndarray)."""
    if node_details is None:
        return []
    try:
        if pd.isna(node_details):
            return []
    except Exception:
        pass
    if isinstance(node_details, np.ndarray):
        return node_details.tolist()
    if isinstance(node_details, (list, tuple)):
        return list(node_details)
    return []


def extract_hops(node_details):
    """Extract a clean list of hop dicts with key fields."""
    hops = []
    for node in iter_nodes(node_details):
        if not isinstance(node, dict):
            continue
        hops.append({
            "ttl": node.get("ttl"),
            "addr": node.get("addr", "*"),
            "asn": node.get("associated_asn"),
            "org": node.get("associated_peeringdb_name") or node.get("associated_org"),
            "ixp": node.get("associated_ixp"),
            "cc": node.get("cc"),
            "place": node.get("place"),
            "lat": node.get("latitude"),
            "lon": node.get("longitude"),
            "rtt": node.get("rtts"),
            "rdns": node.get("rdns_name"),
        })
    return sorted(hops, key=lambda h: h["ttl"] if h["ttl"] is not None else 999)


def extract_as_path(node_details):
    """Extract deduplicated consecutive AS path."""
    path = []
    for node in iter_nodes(node_details):
        if not isinstance(node, dict):
            continue
        asn = node.get("associated_asn")
        if asn is None:
            continue
        try:
            asn = int(asn)
        except Exception:
            continue
        if asn <= 0:
            continue
        if len(path) == 0 or path[-1] != asn:
            path.append(asn)
    return path


def extract_country_path(node_details):
    """Extract deduplicated consecutive country codes."""
    path = []
    for node in iter_nodes(node_details):
        if not isinstance(node, dict):
            continue
        cc = node.get("cc")
        if cc is None or str(cc).strip() == "":
            continue
        cc = str(cc).strip()
        if len(path) == 0 or path[-1] != cc:
            path.append(cc)
    return path


def shorten(name, n=25):
    """Shorten an org name."""
    if not name:
        return "?"
    name = str(name).split(",")[0].split("(")[0].strip()
    return name[:n-1] + "." if len(name) > n else name

## 5. Print a Traceroute

Pick any row and inspect its forward path hop by hop.

In [ ]:
# Pick a specific measurement — change the index to explore others
# Try filtering first: e.g. international paths from Mongolia Telecom
subset = df[(df['dst_site'] == 'icn02') & (df['src_asn'] == 9934)]
# subset = df[(df['dst_site'] == 'uln10076') & (df['src_asn'] == 14593)]  # Starlink domestic
# subset = df_oct[df_oct['dst_site'] == 'lhe152605']  # Lahore (October only)

row = subset.iloc[0] if len(subset) > 0 else df.iloc[0]

print(f"Measurement: {row['id']}")
print(f"Date: {row['partition_date']}")
print(f"Source: {row['client_ip']} — AS{row['src_asn']} ({row['src_asn_name']}) — {row['src_city']}")
print(f"Destination: {row['dst_site']} ({row['dst_city']}, {row['dst_country']}) — AS{row['dst_asn']}")
print(f"RTT: {row['ndt_rtt']:.1f} ms | Throughput: {row['ndt_throughput']:.1f} Mbps | Loss: {row['ndt_loss_rate']:.4f}")
print(f"Forward distance: {row['forward_distance']:.0f} km")
print()

hops = extract_hops(row['forward_updated_node_details'])
as_path = extract_as_path(row['forward_updated_node_details'])
country_path = extract_country_path(row['forward_updated_node_details'])

print(f"AS path ({len(as_path)} ASes): {' → '.join(f'AS{a}' for a in as_path)}")
print(f"Country path: {' → '.join(country_path)}")
print(f"Hops: {len(hops)}")
print()
print(f"{'TTL':>3s}  {'IP':20s}  {'ASN':>7s}  {'Organization':25s}  {'IXP':12s}  {'CC':>2s}  {'City':20s}  {'RTT':>8s}")
print("-" * 110)
for h in hops:
    asn_str = f"AS{h['asn']}" if h['asn'] else "*"
    org = shorten(h['org'], 25) if h['org'] else ""
    ixp = h['ixp'] or ""
    cc = h['cc'] or ""
    place = (h['place'] or "")[:20]
    rtt = f"{h['rtt']:.1f} ms" if h['rtt'] else "?"
    print(f"{h['ttl']:3d}  {h['addr']:20s}  {asn_str:>7s}  {org:25s}  {ixp:12s}  {cc:>2s}  {place:20s}  {rtt:>8s}")

Measurement: ndt-virtual-2zm27_1758441189_0000000000146DAC
Date: 2025-10-01
Source: 202.5.196.32 — AS9934 (Mongolia Telecom) — Ulan Bator-1-MN
Destination: icn02 (Seoul, KR) — AS396982
RTT: 109.5 ms | Throughput: 80.7 Mbps | Loss: 0.0759
Forward distance: 5854 km

AS path (6 ASes): AS15169 → AS2914 → AS10099 → AS58439 → AS9934 → AS10076
Country path: JP → jp → JP → HK → MN
Hops: 12

TTL  IP                        ASN  Organization               IXP           CC  City                       RTT
--------------------------------------------------------------------------------------------------------------
  1  10.29.0.8                   *                                                                            ?
  2  72.14.239.222         AS15169.0  Google LLC                 None          JP  Tokyo-Tokyo-JP         31.6 ms
  3  *                           *                                                                      -1.0 ms
  4  117.103.177.17        AS2914.0  NTT Global IP Ne

## 6. Logical Path View

A HERMES-style multi-row diagram showing IP, ASN, City, and RTT laid out by TTL.
Each dot is color-coded by ASN.

In [ ]:
def plot_logical_path(row, title=None):
    """Plot a single measurement's forward path as a multi-row logical diagram."""
    hops = extract_hops(row['forward_updated_node_details'])
    if not hops:
        print("No hops found.")
        return

    # Assign colors per ASN
    unique_asns = list(dict.fromkeys(h['asn'] for h in hops if h['asn']))
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.D3
    asn_cmap = {asn: palette[i % len(palette)] for i, asn in enumerate(unique_asns)}
    asn_cmap[None] = 'lightgray'

    # Fill gaps in TTL sequence
    ttl_map = {h['ttl']: h for h in hops}
    max_ttl = max(ttl_map.keys())
    ttls = list(range(1, max_ttl + 1))

    fig = go.Figure()
    annotations = []

    rows_config = [
        ('IP',   lambda h: h.get('addr', '*'),                             'circle'),
        ('AS',   lambda h: f"AS{h['asn']}" if h.get('asn') else '*',      'square'),
        ('City', lambda h: (h.get('place') or h.get('cc') or '*')[:20],   'hexagon'),
        ('RTT',  lambda h: f"{h['rtt']:.0f}ms" if h.get('rtt') else '?', 'diamond'),
    ]

    for row_idx, (row_label, text_fn, symbol) in enumerate(rows_config):
        y_val = len(rows_config) - 1 - row_idx
        texts, colors, hovers = [], [], []
        for ttl in ttls:
            h = ttl_map.get(ttl, {})
            texts.append(text_fn(h))
            colors.append(asn_cmap.get(h.get('asn'), 'lightgray'))
            org = shorten(h.get('org'), 20)
            hovers.append(f"TTL {ttl}<br>{h.get('addr','*')}<br>AS{h.get('asn','?')} ({org})<br>{h.get('place','?')}")

        # Connection line for AS row
        if row_label == 'AS':
            fig.add_trace(go.Scatter(
                x=ttls, y=[y_val]*len(ttls), mode='lines',
                line=dict(width=2, color='lightgray'),
                hoverinfo='skip', showlegend=False,
            ))

        fig.add_trace(go.Scatter(
            x=ttls, y=[y_val]*len(ttls),
            mode='markers+text' if row_label in ('IP', 'AS', 'RTT') else 'markers',
            marker=dict(size=10, color=colors, symbol=symbol, line=dict(width=1, color='white')),
            text=texts if row_label in ('AS', 'RTT') else None,
            textposition='top center',
            textfont=dict(size=8 if row_label == 'RTT' else 9),
            hovertext=hovers, hoverinfo='text',
            showlegend=False,
        ))

        # Angled text for City and IP rows
        if row_label in ('City', 'IP'):
            for j, ttl in enumerate(ttls):
                annotations.append(dict(
                    x=ttl, y=y_val, text=texts[j],
                    showarrow=False, textangle=30 if row_label == 'City' else 0,
                    font=dict(size=7 if row_label == 'IP' else 8, color=colors[j]),
                    xanchor='center', yanchor='bottom' if row_label == 'City' else 'top',
                    yshift=8 if row_label == 'City' else -12,
                ))

    # Mark AS transitions
    prev_asn = None
    for ttl in ttls:
        asn = ttl_map.get(ttl, {}).get('asn')
        if asn and asn != prev_asn and prev_asn is not None:
            fig.add_vline(x=ttl - 0.5, line_dash='dot', line_color='rgba(0,0,0,0.15)')
        prev_asn = asn

    # Title
    if title is None:
        src_name = shorten(row.get('src_asn_name'), 20)
        title = f"AS{int(row['src_asn'])} ({src_name}) → {row['dst_site']} ({row['dst_city']})"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=13)),
        annotations=annotations,
        xaxis=dict(title='Hop TTL', tickmode='linear', dtick=1, showgrid=True,
                   gridcolor='rgba(0,0,0,0.04)'),
        yaxis=dict(showticklabels=True, tickvals=list(range(len(rows_config))),
                   ticktext=[r[0] for r in reversed(rows_config)],
                   showgrid=False),
        height=350, width=max(600, len(ttls) * 70),
        margin=dict(l=60, r=30, t=50, b=40),
        plot_bgcolor='white',
    )

    # ASN legend
    for asn in unique_asns:
        name = shorten(next((h['org'] for h in hops if h['asn'] == asn and h['org']), str(asn)), 20)
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=8, color=asn_cmap[asn]),
            name=f'AS{asn} ({name})',
        ))

    fig.show()

In [ ]:
# Example 1: Domestic path (Univision → Ulaanbaatar)
row_domestic = df[df['dst_site'] == 'uln10076'].iloc[0]
plot_logical_path(row_domestic, "Domestic Path — School → Ulaanbaatar")

In [ ]:
# Example 2: International path (→ Seoul via Hong Kong)
row_intl = df[df['dst_site'] == 'icn02'].iloc[0]
plot_logical_path(row_intl, "International Path — School → Seoul")

In [ ]:
# Example 3: Starlink domestic detour through Tokyo
row_starlink = df[(df['dst_site'] == 'uln10076') & (df['src_asn'] == 14593)]
if len(row_starlink) > 0:
    plot_logical_path(row_starlink.iloc[0], "Starlink Domestic Detour — School → Tokyo → Ulaanbaatar")
else:
    print("No Starlink domestic tests in this dataset.")

In [ ]:
# Example 4: Mongolia Telecom → Seoul (complex Japan route, October data)
row_mt = df_oct[(df_oct['dst_site'] == 'icn02') & (df_oct['src_asn'] == 9934)]
if len(row_mt) > 0:
    plot_logical_path(row_mt.iloc[0], "Mongolia Telecom → Seoul (Japan route, Oct 2025)")
else:
    print("No Mongolia Telecom international tests found.")

## 7. Geographic Path View

Plot the physical route on a map. Each hop is a point; lines connect consecutive hops.
A dashed line connects the last observed hop to the client's geolocated position.

In [ ]:
def plot_geographic_path(row, title=None):
    """Plot a single measurement's forward path on a geographic map."""
    hops = extract_hops(row['forward_updated_node_details'])
    geo = [h for h in hops if h['lat'] is not None and h['lon'] is not None
           and not (h['lat'] == 0 and h['lon'] == 0)]
    if not geo:
        print("No geolocated hops.")
        return

    # ASN color mapping
    unique_asns = list(dict.fromkeys(h['asn'] for h in geo if h['asn']))
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.D3
    asn_cmap = {asn: palette[i % len(palette)] for i, asn in enumerate(unique_asns)}

    fig = go.Figure()

    lats = [h['lat'] for h in geo]
    lons = [h['lon'] for h in geo]

    # Hover text
    hover = []
    text_labels = []
    for h in geo:
        asn = h['asn']
        org = shorten(h.get('org'), 20)
        asn_label = f"AS{asn} ({org})" if asn else "?"
        place = h.get('place', '') or h.get('cc', '')
        rtt = f"{h['rtt']:.1f} ms" if h.get('rtt') else "?"
        hover.append(f"<b>TTL {h['ttl']}</b><br>{h['addr']}<br>{asn_label}<br>{place} ({h.get('cc','')})<br>RTT: {rtt}")
        text_labels.append(place[:15] if place else h.get('cc', ''))

    # Path line
    fig.add_trace(go.Scattergeo(
        lat=lats, lon=lons, mode='lines',
        line=dict(width=3, color='steelblue'),
        hoverinfo='skip', name='Path',
    ))

    # Hop markers colored by ASN
    colors = [asn_cmap.get(h['asn'], 'gray') for h in geo]
    sizes = [8] * len(geo)
    sizes[0] = 14   # server end
    sizes[-1] = 12  # last hop before client

    fig.add_trace(go.Scattergeo(
        lat=lats, lon=lons,
        mode='markers+text',
        marker=dict(size=sizes, color=colors, line=dict(width=1, color='white')),
        text=text_labels, textposition='top center', textfont=dict(size=9),
        hovertext=hover, hoverinfo='text', name='Hops',
    ))

    # Dashed line to client
    src_lat, src_lon = row.get('src_lat'), row.get('src_lon')
    if src_lat and src_lon:
        fig.add_trace(go.Scattergeo(
            lat=[geo[-1]['lat'], src_lat], lon=[geo[-1]['lon'], src_lon],
            mode='lines+markers+text',
            line=dict(width=2, color='red', dash='dash'),
            marker=dict(size=[0, 12], symbol=['circle', 'star'], color='red'),
            text=['', f"Client: {row.get('src_city', '?')}"],
            textposition='top center', textfont=dict(size=9, color='red'),
            hoverinfo='skip', name='To client',
        ))

    # ASN legend entries
    for asn in unique_asns:
        name = shorten(next((h['org'] for h in geo if h['asn'] == asn and h.get('org')), str(asn)), 20)
        fig.add_trace(go.Scattergeo(
            lat=[None], lon=[None], mode='markers',
            marker=dict(size=8, color=asn_cmap[asn]),
            name=f'AS{asn} ({name})',
        ))

    if title is None:
        src_name = shorten(row.get('src_asn_name'), 20)
        title = f"AS{int(row['src_asn'])} ({src_name}) → {row['dst_site']} ({row['dst_city']})"

    fig.update_geos(
        showcountries=True, countrycolor='rgb(204,204,204)',
        showland=True, landcolor='rgb(243,243,243)',
        showocean=True, oceancolor='rgb(210,230,250)',
        showlakes=True, lakecolor='rgb(180,210,250)',
        showrivers=True, rivercolor='rgb(170,200,250)',
        showframe=False,
        projection_type='natural earth',
    )
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=13)),
        height=550, margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(yanchor='top', y=0.98, xanchor='left', x=0.01,
                    bgcolor='rgba(255,255,255,0.9)', font=dict(size=10)),
    )
    fig.show()

In [ ]:
# Example 1: Domestic path
plot_geographic_path(row_domestic, "Geographic: Domestic Path → Ulaanbaatar")

In [ ]:
# Example 2: International path → Seoul
plot_geographic_path(row_intl, "Geographic: International Path → Seoul")

In [ ]:
# Example 3: Starlink detour through Tokyo
if len(row_starlink) > 0:
    plot_geographic_path(row_starlink.iloc[0], "Geographic: Starlink → Tokyo → Ulaanbaatar")

In [ ]:
# Example 4: Mongolia Telecom complex route (October)
if len(row_mt) > 0:
    plot_geographic_path(row_mt.iloc[0], "Geographic: Mongolia Telecom → Japan → Seoul (Oct)")

## 8. Combined: Logical + Geographic for Any Measurement

Use this cell to inspect any measurement — just change the filter.

In [ ]:
# ── CHANGE THIS FILTER TO EXPLORE DIFFERENT PATHS ──
# Options:
#   df or df_oct                                  — pick period
#   df['dst_site'] == 'uln10076' / 'icn02'       — domestic or international
#   df['src_asn'] == 17882 / 9934 / 14593 / ...  — pick ISP

my_row = df[(df['dst_site'] == 'icn02')].iloc[0]

print(f"Source: AS{int(my_row['src_asn'])} ({my_row['src_asn_name']}) — {my_row['src_city']}")
print(f"Dest:   {my_row['dst_site']} ({my_row['dst_city']})")
print(f"RTT: {my_row['ndt_rtt']:.1f} ms  |  Throughput: {my_row['ndt_throughput']:.1f} Mbps  |  Loss: {my_row['ndt_loss_rate']:.4f}")
print(f"AS path: {' → '.join(f'AS{a}' for a in extract_as_path(my_row['forward_updated_node_details']))}")
print(f"Countries: {' → '.join(extract_country_path(my_row['forward_updated_node_details']))}")
print()

plot_logical_path(my_row)
plot_geographic_path(my_row)

Source: AS9484 (Mobinet ISP, MobiCom Corporation) — Ulan Bator-1-MN
Dest:   icn02 (Seoul)
RTT: 238.8 ms  |  Throughput: 1.6 Mbps  |  Loss: 0.1337
AS path: AS15169 → AS55805 → AS9484
Countries: HK → MN



## 9. Batch Statistics (Optional)

Quick aggregate stats across all measurements.

In [ ]:
# Compute AS path length and hop count for all measurements
stats = df[['id', 'src_asn', 'src_asn_name', 'dst_site', 'dst_city',
            'ndt_rtt', 'ndt_throughput', 'ndt_loss_rate', 'forward_distance']].copy()

stats['n_hops'] = df['forward_updated_node_details'].apply(lambda nd: len(extract_hops(nd)))
stats['n_as'] = df['forward_updated_node_details'].apply(lambda nd: len(extract_as_path(nd)))
stats['n_countries'] = df['forward_updated_node_details'].apply(lambda nd: len(extract_country_path(nd)))
stats['dst_type'] = stats['dst_site'].apply(lambda s: 'Domestic' if s == 'uln10076' else 'International')

print("=== Average Path Metrics: Domestic vs International ===")
print(stats.groupby('dst_type')[['n_hops', 'n_as', 'forward_distance', 'n_countries',
                                   'ndt_rtt', 'ndt_throughput', 'ndt_loss_rate']].agg(['mean', 'median']).round(2))

=== Average Path Metrics: Domestic vs International ===
              n_hops         n_as        forward_distance           \
                mean median  mean median             mean   median   
dst_type                                                             
Domestic        5.02    4.0  1.64    1.0           317.54    11.47   
International   7.21    7.0  2.87    3.0          3468.25  2923.09   

              n_countries         ndt_rtt        ndt_throughput         \
                     mean median     mean median           mean median   
dst_type                                                                 
Domestic             1.05    1.0  2907.78   14.1         123.90  71.25   
International        2.32    2.0  1908.22  117.5          47.01  31.50   

              ndt_loss_rate         
                       mean median  
dst_type                            
Domestic               0.05   0.01  
International          0.10   0.08  


In [ ]:
# Distribution of AS path lengths
fig = px.histogram(stats, x='n_as', color='dst_type', barmode='group',
                   nbins=10, title='AS Path Length Distribution',
                   labels={'n_as': 'Number of ASes', 'dst_type': 'Destination'})
fig.show()